# Transformer from Scratch: Substring Detection

This notebook implements a **Transformer encoder** from first principles and trains it on a binary sequence classification task: given a string over the alphabet `{c, p, e, n}`, detect whether it contains `"cpen"` as a contiguous substring.

**What's implemented from scratch** (no `torch.nn` convenience modules for architecture internals):
- Character-level tokenizer with one-hot encoding
- Learnable absolute positional encoding (APE)
- Learnable relative positional encoding (RPE) via Toeplitz attention bias
- Multi-head scaled dot-product self-attention
- Transformer encoder layer (Pre-Norm and Post-Norm variants)
- Linear warmup + linear cooldown learning rate scheduler

**What uses PyTorch:** tensor operations, autograd (outer training loop only), `nn.Parameter`, `nn.LayerNorm`, `nn.ReLU`, `F.softmax`, `F.binary_cross_entropy_with_logits`, `Adam` optimizer.

> Originally developed as a solution for an assignment for CPEN 455 (Deep Learning) at UBC.
> Refactored into a standalone module with docstrings and dimension annotations.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add the transformer directory to the path so we can import our module
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from transformer_from_scratch import (
    SubstringDataset,
    Tokenizer,
    AbsolutePositionalEncoding,
    MultiHeadAttention,
    TransformerLayer,
    TransformerModel,
    ModelConfig,
    TrainerConfig,
    Trainer,
    CustomScheduler,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Architecture Overview

The model follows the standard Transformer encoder architecture from [Vaswani et al. (2017)](https://arxiv.org/abs/1706.03762), adapted for binary classification:

```
Input string: "ecpenp"
               │
               ▼
   Tokenizer ──► one-hot matrix  (n_tokens × 5)
               │
   Prepend [CLS] token
               │
               ▼
   Linear Encoder  W_enc ──► (n_tokens × d_model)
               │
   Positional Encoding (APE or RPE)
               │
               ▼
   ┌─────────────────────────┐
   │   Transformer Layer ×N  │
   │  ┌───────────────────┐  │
   │  │ Multi-Head Self-  │  │
   │  │ Attention         │  │
   │  └───────────────────┘  │
   │  + Residual + LayerNorm │
   │  ┌───────────────────┐  │
   │  │ Feed-Forward      │  │
   │  │ ReLU(X·W1)·W2     │  │
   │  └───────────────────┘  │
   │  + Residual + LayerNorm │
   └─────────────────────────┘
               │
               ▼
   Linear Decoder  W_dec ──► (n_tokens × 1)
               │
   Extract [CLS] output (position 0)
               │
               ▼
   Binary cross-entropy loss ──► label ∈ {0, 1}
```

The key insight: by prepending a `[CLS]` token that attends to every other position via self-attention, the model can learn to aggregate information across the full sequence into a single classification decision at position 0.

## Component 1: Tokenizer

The tokenizer converts raw strings into one-hot matrices. Each character maps to a vocabulary index, and each index becomes a one-hot row vector.

**Vocabulary:** `[CLS]=0, c=1, p=2, e=3, n=4` → `d_voc = 5` 
for a string of length `n` with `[CLS]` prepended, the output is an `(n+1) × 5` matrix.

In [ ]:
tok = Tokenizer()

# Tokenize a sample string with [CLS] prepended
sample = "cpen"
one_hot = tok.tokenize_string(sample, add_cls_token=True)
print(f"Input string: '{sample}'")
print(f"Output shape: {one_hot.shape}  (5 tokens × 5 vocab size)")
print(f"\nOne-hot matrix (rows=[CLS], c, p, e, n):")
print(one_hot)

# Verify: each row sums to 1 (valid one-hot)
assert one_hot.sum(dim=1).allclose(torch.ones(5)), "One-hot rows should sum to 1"

# Batch tokenization
batch = tok.tokenize_string_batch(["cpen", "ecpe", "nnnn"], add_cls_token=True)
print(f"\nBatch shape: {batch.shape}  (3 strings × 5 tokens × 5 vocab)")

## Component 2: Positional Encoding

Transformers process all tokens in parallel (unlike RNNs), so they have **no inherent notion of token order**. Positional encoding injects position information.

### Absolute Positional Encoding (APE)

A learnable matrix $W_{pos} \in \mathbb{R}^{\text{max\_len} \times d_{\text{model}}}$ where the $i$-th row is the embedding for position $i$. Applied by element-wise addition:

$$\text{out}_i = x_i + W_{pos}[i]$$

### Relative Positional Encoding (RPE)

Instead of adding position to embeddings, RPE adds a **bias to attention scores** based on the *relative distance* between token pairs. For each head $h$, a learnable Toeplitz matrix $M_h$ with entry $(i,j) = m_{i-j,h}$ is added inside the softmax:

$$\text{head}_h = \text{softmax}\left(\frac{Q_h K_h^T + M_h}{\sqrt{d_h}}\right) V_h$$

The Toeplitz structure means only $2n-1$ parameters encode all pairwise relative distances, and the encoding is **translation-invariant**: it cares about how far apart tokens are, not their absolute positions.

In [ ]:
# Demonstrate APE: position embeddings shift each token's representation
ape = AbsolutePositionalEncoding(d_model=64)

# Two identical sequences at different batch positions get the same positional shift
x = torch.zeros(1, 6, 64)  # 6 tokens, all zeros
out = ape(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"\nPositional embeddings are added — output at position 0 vs position 5:")
print(f"  Position 0 norm: {out[0, 0].norm().item():.4f}")
print(f"  Position 5 norm: {out[0, 5].norm().item():.4f}")
print(f"  Are they different? {not torch.allclose(out[0, 0], out[0, 5])}")

## Component 3: Multi-Head Self-Attention

The core of the Transformer. Each head computes scaled dot-product attention independently, then results are concatenated and linearly projected.

**Per head $h$:**
$$\text{head}_h = \text{softmax}\left(\frac{(X W_{Q,h})(X W_{K,h})^T}{\sqrt{d_h}}\right)(X W_{V,h})$$

where $W_{Q,h}, W_{K,h}, W_{V,h} \in \mathbb{R}^{d_{\text{model}} \times d_h}$ and $d_h = d_{\text{model}} / H$.

**Combined:**
$$\text{Attention}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_H) \cdot W_O$$

The $\sqrt{d_h}$ scaling prevents dot products from growing too large as dimensionality increases, which would push softmax into saturated regions with near-zero gradients.

**Why multiple heads?** Each head can learn to attend to different types of relationships (e.g., one head might learn adjacency patterns, another might learn long-range dependencies).

In [ ]:
# Multi-head attention without RPE
mha = MultiHeadAttention(d_model=64, n_heads=4, rpe=False)
x = torch.randn(2, 6, 64)  # batch=2, seq_len=6, d_model=64
out = mha(x, x, x)  # self-attention: key=query=value=x
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"d_h per head: {mha.d_h}")
print(f"Number of heads: {mha.n_heads}")

# Multi-head attention with RPE
mha_rpe = MultiHeadAttention(d_model=64, n_heads=4, rpe=True)
out_rpe = mha_rpe(x, x, x)
print(f"\nWith RPE — output shape: {out_rpe.shape}")
print(f"RPE parameters per head: {mha_rpe.rpe_w[0].shape[0]} (2*MAX_LEN+1)")

## Component 4: Transformer Layer

Each layer wraps self-attention and a feed-forward network with residual connections and layer normalization. Two variants:

**Pre-Norm** (normalize *before* each sublayer — generally more stable for deep models):
$$x = x + \text{Attention}(\text{LN}(x))$$
$$x = x + \text{FC}(\text{LN}(x))$$

**Post-Norm** (original Transformer paper — normalize *after*):
$$x = \text{LN}(x + \text{Attention}(x))$$
$$x = \text{LN}(x + \text{FC}(x))$$

The feed-forward network expands to 4× the model dimension and contracts back:
$$\text{FC}(X) = \text{ReLU}(X W_1) W_2$$
where $W_1 \in \mathbb{R}^{d_{\text{model}} \times 4 d_{\text{model}}}$, $W_2 \in \mathbb{R}^{4 d_{\text{model}} \times d_{\text{model}}}$.

In [ ]:
# Compare Pre-Norm and Post-Norm
x = torch.randn(2, 6, 64)

layer_pre = TransformerLayer(d_model=64, n_heads=4, prenorm=True, rpe=False)
layer_post = TransformerLayer(d_model=64, n_heads=4, prenorm=False, rpe=False)

out_pre = layer_pre(x)
out_post = layer_post(x)

print(f"Input shape:       {x.shape}")
print(f"Pre-Norm output:   {out_pre.shape}")
print(f"Post-Norm output:  {out_post.shape}")
print(f"\nPre-Norm output std:  {out_pre.std().item():.4f}")
print(f"Post-Norm output std: {out_post.std().item():.4f}")

## Component 5: Full Transformer Model

The complete model chains:
1. **Encoder** ($W_{enc}$): projects one-hot tokens from $d_{voc}=5$ into $d_{model}$
2. **Positional encoding**: APE (additive) or RPE (attention bias)
3. **$N$ Transformer layers**: stacked self-attention + feed-forward
4. **Decoder** ($W_{dec}$): projects from $d_{model}$ to $d_{out}=1$

For classification, only the `[CLS]` token output at position 0 is used.

In [ ]:
cfg = ModelConfig(d_model=64, n_heads=4, n_layers=2)
model = TransformerModel(cfg)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model config: d_model={cfg.d_model}, n_heads={cfg.n_heads}, "
      f"n_layers={cfg.n_layers}, pos_enc={cfg.pos_enc_type}")
print(f"Total parameters: {n_params:,}")

# Forward pass with a batch of one-hot inputs
x = torch.randn(2, 6, 5)  # batch=2, seq_len=6, d_voc=5
out = model(x)
print(f"\nInput shape:  {x.shape}  (B, seq_len, d_voc)")
print(f"Output shape: {out.shape}  (B, seq_len, d_out)")
print(f"[CLS] logit:  {out[:, 0, :].squeeze(-1)}  (position 0)")

## Component 6: Learning Rate Scheduler

Transformers benefit from **warmup + cooldown** scheduling:

- **Warmup** (steps 0 → `warmup_steps`): LR linearly increases from 0 to `base_lr`.
  This stabilizes early training when gradients are noisy and weights are random.
- **Cooldown** (steps `warmup_steps` → `total_steps`): LR linearly decreases from
  `base_lr` back to 0. This fine-tunes the model with progressively smaller updates.

In [ ]:
# Visualize the scheduler profile
dummy_model = TransformerModel(ModelConfig(d_model=64, n_heads=4, n_layers=1))
optimizer = torch.optim.Adam(dummy_model.parameters(), lr=0.003)
scheduler = CustomScheduler(optimizer, total_steps=5000, warmup_steps=1000)

lrs = []
for step in range(5000):
    optimizer.step()
    scheduler.step()
    lrs.append(optimizer.param_groups[0]["lr"])

plt.figure(figsize=(8, 3))
plt.plot(lrs, linewidth=1.5)
plt.xlabel("Training step")
plt.ylabel("Learning rate")
plt.title("Linear Warmup + Linear Cooldown Schedule")
plt.axvline(x=1000, color="gray", linestyle="--", alpha=0.5, label="End of warmup")
plt.legend()
plt.tight_layout()
plt.show()

## Dataset Preparation

We generate three datasets of random strings over `{c, p, e, n}`:
- **Train:** 10,000 samples (balanced: 50% contain "cpen", 50% don't)
- **Validation:** 1,000 samples
- **Test:** 1,000 samples

Each string has length 16. With the `[CLS]` token prepended, the sequence length is 17.

In [ ]:
STR_LEN = 16

train_dataset = SubstringDataset(seed=1, dataset_size=10_000, str_len=STR_LEN)
val_dataset = SubstringDataset(seed=2, dataset_size=1_000, str_len=STR_LEN)
test_dataset = SubstringDataset(seed=3, dataset_size=1_000, str_len=STR_LEN)

# Inspect a few samples
print(f"Dataset sizes: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
print(f"String length: {STR_LEN} characters (+1 for [CLS] = {STR_LEN + 1} tokens)")
print(f"\nSample strings:")
for i in range(6):
    s, label = train_dataset[i]
    has_cpen = "✓ contains 'cpen'" if label == 1 else "✗ no 'cpen'"
    print(f"  '{s}'  →  label={label}  ({has_cpen})")

## Training

We train the full Transformer model with:
- **Model:** 4 layers, d_model=256, 4 attention heads, Pre-Norm, APE
- **Optimizer:** Adam with base lr=0.003
- **Scheduler:** Linear warmup (1000 steps) + linear cooldown (total 5000 steps)
- **Loss:** Binary cross-entropy with logits on the `[CLS]` output

The task is synthetic and well-structured, so the model should converge to near-perfect accuracy within a few thousand steps.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

model = TransformerModel(ModelConfig())
trainer = Trainer(model, TrainerConfig(device=device))

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training for {trainer.cfg.train_steps} steps...\n")

trainer.train(train_dataset, val_dataset)

## Evaluation

In [ ]:
test_loss, test_acc = trainer.evaluate_dataset(test_dataset)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

## Training Curves

Let's re-train with metric logging to visualize the learning dynamics.

In [ ]:
# Re-train with metric logging
from torch.utils.data import DataLoader

model2 = TransformerModel(ModelConfig())
model2 = model2.to(device)
tokenizer = Tokenizer()

optimizer = torch.optim.Adam(model2.parameters(), lr=0.003)
scheduler = CustomScheduler(optimizer, total_steps=5000, warmup_steps=1000)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=256)

# Logging
log_steps, log_train_loss, log_val_loss, log_val_acc = [], [], [], []

for step in range(5000):
    model2.train()
    strings, y = next(iter(train_loader))
    x = tokenizer.tokenize_string_batch(strings).to(device)
    y = y.to(device).float()

    optimizer.zero_grad()
    logits = model2(x)
    cls_logits = logits[:, 0, :].squeeze(-1)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(cls_logits, y)
    loss.backward()
    optimizer.step()
    scheduler.step()

    if step % 50 == 0:
        # Validation
        model2.eval()
        with torch.no_grad():
            val_losses, val_accs, val_n = 0.0, 0.0, 0
            for vs, vy in DataLoader(val_dataset, batch_size=256):
                vx = tokenizer.tokenize_string_batch(vs).to(device)
                vy = vy.to(device).float()
                vlogits = model2(vx)[:, 0, :].squeeze(-1)
                vloss = torch.nn.functional.binary_cross_entropy_with_logits(vlogits, vy)
                vacc = ((torch.sigmoid(vlogits) > 0.5).float() == vy).float().mean()
                bs = vx.size(0)
                val_losses += vloss.item() * bs
                val_accs += vacc.item() * bs
                val_n += bs

        log_steps.append(step)
        log_train_loss.append(loss.item())
        log_val_loss.append(val_losses / val_n)
        log_val_acc.append(val_accs / val_n)

print("Training complete.")
print(f"Final val accuracy: {log_val_acc[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(log_steps, log_train_loss, label="Train loss", alpha=0.8)
axes[0].plot(log_steps, log_val_loss, label="Val loss", alpha=0.8)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss (BCE)")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(log_steps, log_val_acc, color="green", linewidth=2)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].axhline(y=1.0, color="gray", linestyle="--", alpha=0.4, label="Perfect")
axes[1].set_ylim(0.4, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Final test evaluation on the logged model
model2.eval()
test_correct, test_total = 0, 0
test_loss_sum = 0.0

with torch.no_grad():
    for ts, ty in DataLoader(test_dataset, batch_size=256):
        tx = tokenizer.tokenize_string_batch(ts).to(device)
        ty_d = ty.to(device).float()
        tlogits = model2(tx)[:, 0, :].squeeze(-1)
        tloss = torch.nn.functional.binary_cross_entropy_with_logits(tlogits, ty_d)
        tpreds = (torch.sigmoid(tlogits) > 0.5).float()
        test_correct += (tpreds == ty_d).sum().item()
        test_total += tx.size(0)
        test_loss_sum += tloss.item() * tx.size(0)

print(f"Test Loss:     {test_loss_sum / test_total:.4f}")
print(f"Test Accuracy: {test_correct / test_total:.4f}  ({test_correct}/{test_total})")

## Some Things I Learned

Key insights from implementing a Transformer from scratch:

1. **Attention is just weighted averaging.** Strip away the notation and each head computes a data-dependent weighted average of value vectors via softmax over query-key dot products. The power comes from *learning* which pairs to weight highly.

2. **Positional encoding is essential, not optional.** Without it, the model treats "cpen" and "npec" identically: self-attention is permutation-equivariant by design. APE and RPE solve this in fundamentally different ways: APE stamps each position with a fixed identity; RPE encodes the *distance* between positions directly in the attention scores.

3. **The `[CLS]` token is a learned aggregation slot.** It has no inherent meaning, it's a dummy token whose self-attention weights learn to pool information from the entire sequence into a single representation suitable for classification.

4. **Warmup matters for Transformers.** Without warmup, early gradient updates are large and erratic (Adam's second-moment estimates haven't stabilized), which can cause the attention weights to collapse. Linear warmup avoids this by starting with tiny learning rates.

5. **Pre-Norm vs. Post-Norm is a real design choice.** Pre-Norm places normalization *before* each sublayer, which stabilizes gradients and typically allows training deeper models. The original Transformer used Post-Norm, but most modern architectures default to Pre-Norm (I think)